# Pawnee National Grassland Land Swap
- **Objective:** For this notebook, the observations of 5 grasses, including blue grama(Bouteloua gracilis), buffalograss(Bouteloua dactyloides), sideoats grama(Bouteloua curtipendula), western wheatgrass(Pascopyrum smithii), and needle-and-thread (Hesperostipa comata), are pulled from [GBIF](https://www.gbif.org/) for each parcel within the Pawnee National Grassland boundary. A "ecological value" for these species will be generated for each parcel, and in the `parcel_matrix` notebook these ecological values will be appended to identify best land swaps.
- **Author:** Kayleigh Ward
- **Author:**
- **Code review and/or edits:** Kayleigh Ward
- **Date:** April 9, 2026
- **Last date of revision:** April 9, 2026

---
### 🛠️ Prerequisites & Setup
* **Libraries:** ex: `pandas`, `scikit-learn`, `matplotlib`
* **Data Sources:** ex: `/data/raw/dataset.csv`
* **Related Notebooks:** Must run the `boundaries` notebook prior to running this notebook and pull stored values.

### 🏗️ Methodology
1.  **Data Loading:** Login to GBIF and then download one GBIF species file, extract it, and append it to the GBIF grasses DataFrame. 
2.  **Cleaning:** Clip the GBIF grasses observations to the Pawnee National Grassland boundary
3.  **Visualization:** Plot the species distribution for both each grass to confirm that the clip was successful.

### ⚡ Troubleshooting/Notes
* Example Note: If data updates, run `01_refresh.ipynb` first.

# Libraries

In [ ]:
### import libraries
import os
import pathlib
import requests
import zipfile
import geopandas as gpd
import holoviews as hv
import hvplot.pandas
from glob import glob
import time
import pandas as pd

### gbif packages
import pygbif.occurrences as occ
import pygbif.species as species
from getpass import getpass

# Directories

In [ ]:
### set up root file path
# Walk up from the current directory to find the repo root (contains .git)
_cwd = pathlib.Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / '.git').exists()),
    _cwd
)
os.chdir(repo_root)

data_dir = os.path.join(repo_root, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Repo root: {repo_root}')

# GBIF Login for data download

In [ ]:
### reset credentials
reset_credentials = False

### make dictionary for GBIF username and pass
credentials = dict(
    GBIF_USER=(input, 'GBIF username:'),
    GBIF_PWD=(getpass, 'GBIF password'),
    GBIF_EMAIL=(input, 'GBIF email'),
)

### loop through credentials and enter them
for env_variable, (prompt_func, prompt_text) in credentials.items():

    if reset_credentials and (env_variable in os.environ):
        os.environ.pop(env_variable)

    if not env_variable in os.environ:
        os.environ[env_variable] = prompt_func(prompt_text)

## Download GBIF data for Pawnee National Grassland grasses

1. Create a helper function to pull data from GBIF
2. Define the species to download
3. Download data for the following grass species:
- blue grama(Bouteloua gracilis)
- buffalograss(Bouteloua dactyloides)
- sideoats grama(Bouteloua curtipendula)
- western wheatgrass(Pascopyrum smithii)
- needle-and-thread (Hesperostipa comata)

In [ ]:
### function to download GBIF data for five grasses
def download_gbif_grass_species(species_name, folder_name, data_dir):
    """
    Download one GBIF species file, extract it, and read it into a DataFrame.
    """
    ### search species in GBIF backbone
    species_info = species.name_backbone(name=species_name)

    ### get the usage key
    species_key = species_info["usageKey"]
    print(f"{species_name}: {species_key}")

    ### set species directory
    gbif_grasses_dir = os.path.join(data_dir, folder_name)
    os.makedirs(gbif_grasses_dir, exist_ok=True)

    ### make a file path pattern for data
    gbif_grasses_pattern = os.path.join(gbif_grasses_dir, "*.csv")

    ### download it once
    if not glob(gbif_grasses_pattern):

        ### submit query
        gbif_query = occ.download([
            f"speciesKey = {species_key}",
            "hasCoordinate = True",
        ])

        download_key = gbif_query[0]

        ### wait for the download to build
        wait = occ.download_meta(download_key)["status"]
        while wait != "SUCCEEDED":
            if wait in {"CANCELLED", "KILLED", "FAILED"}:
                raise RuntimeError(f"GBIF download failed for {species_name}: {wait}")
            time.sleep(5)
            wait = occ.download_meta(download_key)["status"]

        ### download data
        download_info = occ.download_get(
            download_key,
            path=gbif_grasses_dir
        )

        ### unzip the file
        with zipfile.ZipFile(download_info["path"]) as download_zip:
            download_zip.extractall(path=gbif_dir)

    ### find csv file path
    gbif_grasses_path = glob(gbif_pattern)[0]

    ### read csv
    gbif_grasses_df = pd.read_csv(
        gbif_grasses_path,
        delimiter="\t",
        low_memory=False
    )

    return gbif_grasses_df, gbif_grasses_path, species_key

In [ ]:
### define the species to download
species_to_download = [
    {
        "species_name": "Bouteloua gracilis",
        "folder_name": "gbif_blue_grama"
    },
    {
        "species_name": "Bouteloua dactyloides",
        "folder_name": "gbif_buffalograss"
    }
    {
        "species_name": "Bouteloua curtipendula",
        "folder_name": "gbif_sideoats_grama"
    }
       {
        "species_name": "Pascopyrum smithii",
        "folder_name": "gbif_western_wheatgrass"
    }
       {
        "species_name": "Hesperostipa comata",
        "folder_name": "gbif_needle_and_thread"
    }
]

### create an empty dictionary
gbif_grasses_data = {}

### loop through the download_gbif_species download
for item in species_to_download:
    gbif_grasses_df, gbif_grasses_path, species_key = download_gbif_species(
        species_name=item["species_name"],
        folder_name=item["folder_name"],
        data_dir=data_dir
    )

    gbif_grasses_data[item["species_name"]] = {
        "df": gbif_grasses_df,
        "path": gbif_grasses_path,
        "species_key": species_key
    }

    print(f"\nLoaded {item['species_name']}")
    print(f"Path: {gbif_grasses_path}")
    print(gbif_grasses_df.head())

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1631097267.py, line 7)